# 🚀 Running this Notebook in Google Colab

### Prerequisites & Kernel Setup:
This notebook contains **Scala** code. To run it smoothly in Google Colab:
1. **Method 1 (Native Scala via Almond Kernel - Recommended):**
   - Run the setup cell below. It installs the **Almond Scala kernel** into your Colab VM in ~30 seconds.
   - Once it completes, click **Runtime** ➔ **Manage Sessions** ➔ terminate the active session (or reload your browser tab).
   - Reconnect to the notebook and all Scala cells will execute natively!
2. **Method 2 (Zero-Setup Companion Script):**
   - If you prefer not to switch kernels, each exercise also has a standalone `.scala` file included in this directory. You can run it via the bash execution cell at the end of the notebook!


In [ ]:
# [Colab Setup Cell] Run this to install Almond Scala Kernel in Google Colab
# (Only needed once per session)
!curl -Lo coursier https://git.io/coursier-cli && chmod +x coursier
!./coursier launch --fork almond:0.13.14 --scala 2.13.12 -- --install
println("Almond Scala Kernel installed successfully! Reload tab if prompted.")


# Exercise 15: Handling Collisions and Resolving Conflicts in HashMaps

## Problem Statement:
1. Create a `HashMap` with keys representing student names and values representing their grades, then insert multiple entries including a duplicate key with a different grade.
2. Write a function to merge two HashMaps, resolving conflicts by taking the **higher grade**.
3. Write a function to handle collisions by **chaining**, using `List`s to store multiple values for a single key.

---

## Theoretical Deep Dive: Hash Tables and Collision Resolution
- **Standard HashMap Semantics**:
  - In a standard key-value map, each key is uniquely associated with at most one value.
  - When you insert an entry with an existing key, the previous value is silently overwritten.
- **Conflict Resolution via Value Merging**:
  - When merging two datasets (such as exam 1 and exam 2 results), key clashes occur.
  - Rather than arbitrarily overwriting, we specify a reduction rule: `math.max(gradeA, gradeB)` to preserve the student's highest performance.
- **Collision Resolution via Separate Chaining**:
  - Separate chaining stores values within bucket collections (e.g. `Map[String, List[Int]]`).
  - When a duplicate key arrives (e.g. a student taking multiple quizzes), the new score is appended to the list rather than overwriting previous scores, preserving the full submission history.


In [ ]:
import scala.collection.mutable

// 1. Create a mutable HashMap with student names and grades
val gradeMap = mutable.HashMap[String, Int]()
gradeMap.put("Alice", 85)
gradeMap.put("Bob", 78)
gradeMap.put("Charlie", 92)

println("--- Initial HashMap Entries ---")
gradeMap.foreach { case (name, grade) => println(s"  $name -> $grade") }

// Insert a duplicate key "Alice" with a different grade (e.g. 95)
println("\nInserting duplicate key 'Alice' with new grade 95...")
val oldGrade = gradeMap.put("Alice", 95)

println(s"Replaced previous grade: $oldGrade")
println("--- HashMap After Duplicate Key Insertion (Overwritten) ---")
gradeMap.foreach { case (name, grade) => println(s"  $name -> $grade") }


--- Initial HashMap Entries ---
  Alice -> 85
  Bob -> 78
  Charlie -> 92

Inserting duplicate key 'Alice' with new grade 95...
Replaced previous grade: Some(85)
--- HashMap After Duplicate Key Insertion (Overwritten) ---
  Alice -> 95
  Bob -> 78
  Charlie -> 92


In [ ]:
// 2. Function to merge two HashMaps, resolving conflicts by taking the HIGHER grade
def mergeResolvingHigherGrade(map1: Map[String, Int], map2: Map[String, Int]): Map[String, Int] = {
  val allKeys = map1.keySet ++ map2.keySet
  
  allKeys.map { key =>
    val resolvedGrade = (map1.get(key), map2.get(key)) match {
      case (Some(g1), Some(g2)) => math.max(g1, g2) // Both maps have key -> pick max!
      case (Some(g1), None)     => g1               // Only in map1
      case (None, Some(g2))     => g2               // Only in map2
      case _                    => 0
    }
    key -> resolvedGrade
  }.toMap
}

// Sample test data for two classes / exam sessions
val class1: Map[String, Int] = Map("Alice" -> 85, "Bob" -> 90, "David" -> 74)
val class2: Map[String, Int] = Map("Alice" -> 92, "Bob" -> 82, "Emma" -> 88)

println(s"Class 1 Grades: $class1")
println(s"Class 2 Grades: $class2")

val resolvedMap = mergeResolvingHigherGrade(class1, class2)
println(s"\nMerged Map (Conflicts resolved by higher grade): $resolvedMap")


Class 1 Grades: Map(Alice -> 85, Bob -> 90, David -> 74)
Class 2 Grades: Map(Alice -> 92, Bob -> 82, Emma -> 88)

Merged Map (Conflicts resolved by higher grade): Map(Alice -> 92, Bob -> 90, David -> 74, Emma -> 88)


In [ ]:
// 3. Function to handle collisions by chaining using lists for multiple values
def insertWithChaining(
  map: Map[String, List[Int]],
  student: String,
  grade: Int
): Map[String, List[Int]] = {
  val existingGrades = map.getOrElse(student, List.empty[Int])
  // Append the new grade to the student's grade list
  map + (student -> (existingGrades :+ grade))
}

// Build chained map from a sequence of student submissions containing duplicates
def buildChainedMap(submissions: Seq[(String, Int)]): Map[String, List[Int]] = {
  submissions.foldLeft(Map.empty[String, List[Int]]) { (accMap, submission) =>
    insertWithChaining(accMap, submission._1, submission._2)
  }
}

// Sequence containing multiple submissions for Alice and Bob
val allSubmissions = Seq(
  ("Alice", 85),
  ("Bob", 78),
  ("Charlie", 92),
  ("Alice", 95),  // Alice Quiz 2
  ("Bob", 88),    // Bob Quiz 2
  ("Alice", 90),  // Alice Quiz 3
  ("David", 81)
)

val chainedResults = buildChainedMap(allSubmissions)

println("--- Chained Hash Table Results (All Historical Grades Preserved) ---")
chainedResults.foreach { case (student, grades) =>
  val average = grades.sum.toDouble / grades.length
  println(f"Student: $student%-8s | All Scores: ${grades.mkString("[", ", ", "]")}%-18s | Best: ${grades.max}%2d | Avg: $average%.2f")
}


--- Chained Hash Table Results (All Historical Grades Preserved) ---
Student: Alice    | All Scores: [85, 95, 90]       | Best: 95 | Avg: 90.00
Student: Bob      | All Scores: [78, 88]           | Best: 88 | Avg: 83.00
Student: Charlie  | All Scores: [92]               | Best: 92 | Avg: 92.00
Student: David    | All Scores: [81]               | Best: 81 | Avg: 81.00
